In [21]:
# ============================================================
# Project root & path handling
# ------------------------------------------------------------
# Default: working directory
# ============================================================

from pathlib import Path
import os

# Determine project root
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "..")).expanduser().resolve()

# Convenience function for building repo-relative paths
def p(rel_path):
    
    """
    Build an absolute path from a path relative to the project root.
    """
    return PROJECT_ROOT / rel_path

print("PROJECT_ROOT set to:", PROJECT_ROOT)

PROJECT_ROOT set to: /Users/jeremy/Development/Projects/IGVF CVFG project/Source/IGVF-cvfg-pillar-project


In [22]:
# ============================================================
# Create temporary and output directories 
# ============================================================

DATA_DIR = p("data")
TMP_DIR = p("tmp")
OUT_DIR = p("outputs")

TMP_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

In [23]:
import pandas as pd

pp = pd.read_csv(OUT_DIR/"integrated_variant_effect_dataset.tsv.gz", sep = '\t')

/var/folders/h_/3xfwqd9n4cb16jb_bj7q7w780000gn/T/ipykernel_5736/2578885430.py:3: DtypeWarning: Columns (3,5,6,9,10,14,15,17,24,25,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,67,69,70,71,72,73,74,75,76,77,78,79,80,81,84,90) have mixed types. Specify dtype option on import or set low_memory=False.
  pp = pd.read_csv(OUT_DIR/"integrated_variant_effect_dataset.tsv.gz", sep = '\t')


In [24]:
# JS 20260713 Add empty Flag column
pp['Flag'] = ''

In [ ]:
#flag CHEK2 variants that need to be filetered for any clinical analysis

chek2 = pd.read_excel(DATA_DIR/"raw_mave_data/CHEK2_Gebbia_2024.xlsx", header = 0)

chek2['score'] = chek2['score'].astype(object)

tmp = pd.merge(pp, chek2, left_on = ['hgvs_p', 'auth_reported_score'],
               right_on = ['hgvs_pro','score'], how = 'left')

import numpy as np
tmp['Flag'] = np.where(tmp['Filter_CI'] == 1, '*', tmp['Flag'])

pp = tmp.drop(columns = ['aaChange', 'hgvs_pro',
       'type', 'score', 'error', 'LLR', 'LLR_strength', 'Filter_CI',
       'Filter_Hypercomplement'])

In [26]:
#find and mark splice variants 
import numpy as np

cond2 = pp[
    ['spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL']
].ge(0.2).any(axis=1)

cond3 = pp['simplified_consequence'] == 'splice_site_variant'

pp['splice_variant'] = np.where(cond2|cond3, 'Yes', 'No')

In [27]:
#catch all splice variants in cDNA assays and broadcast to amino acid group
pp['Ref_seq_transcript_ID_stripped'] = pp['RefSeq Transcript ID'].str.replace(r'\.\d+$', '', regex=True)

pp['aa_pos'] = pd.to_numeric(pp['aa_pos'], errors='coerce')
pp['aa_ref'] = pp['aa_ref'].astype(str)
pp['aa_alt'] = pp['aa_alt'].astype(str)
pp['Gene'] = pp['Gene'].astype(str)
pp['Ref_seq_transcript_ID_stripped'] = pp['Ref_seq_transcript_ID_stripped'].astype(str)

group_cols_aa = ['Gene', 'aa_ref', 'aa_pos', 'aa_alt','Ref_seq_transcript_ID_stripped']

def splice_var(series):
    sigs = set(series.dropna())
    
    if len(sigs) == 0:
        return ""
    if len(sigs) == 1:
        return next(iter(sigs))
    if len(sigs) > 1:
        return "Yes"

mask_aa = pp['nucleotide_or_aa'] == 'aa'
        
pp.loc[mask_aa, "splice_var_amino"] = (
    pp.loc[mask_aa]
        .groupby(group_cols_aa)["splice_variant"]
        .transform(splice_var)
)  

In [28]:
#add in MSH2 scott thresholds , clinically relevant thresholds 

MSH2_scott = pp[pp['Dataset'] == 'MSH2_Jia_2021']

# JS 20260713 We no longer use the ID column, which is replaced by mavedb_variant_urn
# MSH2_scott['ID'] = MSH2_scott['ID'].str.replace("Jia_2021", "Scott_2022", regex=False)

MSH2_scott['Dataset'] = 'MSH2_Scott_2022'

pp_2 = pd.concat([pp, MSH2_scott])

/var/folders/h_/3xfwqd9n4cb16jb_bj7q7w780000gn/T/ipykernel_5736/2089249397.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  MSH2_scott['Dataset'] = 'MSH2_Scott_2022'


In [29]:
#dictionary mapping all functional classifications given by author to a standardized class

func_class = {'BAP1_Waters_2024': {'depleted':'Abnormal','unchanged':'Normal','enriched':'Not specified'},
             'BRCA2_Hu_2024': {'Abnormal': 'Abnormal', 'Normal': 'Normal', 'Intermediate': 'Indeterminate'},
             'CRX_Shepherdson_2024': {'low_activity': 'Abnormal','non-significant': 'Normal',
                                      'high_activity': 'Not specified'},
             'DDX3X_Radford_2023': {'fast depleting': 'Abnormal', 'slow depleting': 'Abnormal',
                                              'unchanged': 'Normal', 'enriched': 'Not specified'},
             'FKRP_Ma_2024': {'damaging_severe': 'Abnormal', 'damaging_mild': 'Abnormal', 
                              'damaging_intermediate': 'Abnormal', 'functional': 'Normal'},
             'JAG1_Gilbert_2024': {'Abnormal': 'Abnormal', 'Likely abnormal': 'Abnormal','Normal': 'Normal'},
             'KCNE1_Muhammad_2024_trafficking_WT_background_DN': {'Loss': 'Abnormal','Possible':'Abnormal','Partial':'Abnormal',
                                                   'Normal':'Normal','Gain':'Not specified','PossibleGain':'Not specified'},
             'KCNE1_Muhammad_2024_trafficking': {'Loss': 'Abnormal','Possible':'Abnormal','Partial':'Abnormal',
                                                   'Normal':'Normal','Gain':'Not specified','PossibleGain':'Not specified'},
             'KCNE1_Muhammad_2024_potassium_flux': {'Loss': 'Abnormal','Possible':'Abnormal','Partial':'Abnormal',
                                                   'Normal':'Normal','Gain':'Not specified','PossibleGain':'Not specified'},
             'LARGE1_Ma_2024': {'damaging':'Abnormal','functional': 'Normal'},
             'NDUFAF6_Sung_2024': {'abnormal': 'Abnormal','normal':'Normal','uncertain': 'Indeterminate'},
             'OTC_Lo_2023': {'Amorphic':'Abnormal','Unimpaired':'Normal', 'Hypomorphic':'Not specified'},
             'RAD51C_Olvera-León_2024': {'fast depleted': 'Abnormal','slow depleted': 'Abnormal',
                                                       'unchanged': 'Normal','enriched':'Not specified'},
             'RHO_Wan_2019':{'low': 'Abnormal','very low': 'Abnormal','high': 'Normal','indeterminate': 'Indeterminate'},
             'SCN5A_Glazer_2020': {'LOF':'Abnormal','possiblyLOF':'Abnormal','possiblyWT':'Normal','WT':'Normal',
                                  'GOF': 'Not specified','possiblyGOF':'Not specified'},
             'SCN5A_Ma_2024':{'severe LOF': 'Abnormal', 'moderate LOF': 'Abnormal','normal': 'Normal'},
             'SGCB_Li_2023': {'Non-Functional': 'Abnormal','Functional': 'Normal'},
             'TP53_Fayer_2021_meta': {'Functionally abnormal': 'Abnormal','Functionally normal': 'Normal'},
             'VHL_Buckley_2024': {'LOF1': 'Abnormal','LOF2': 'Abnormal','Neutral': 'Normal', 'Intermediate': 'Indeterminate'},
             'BARD1_IGVF': {'functionally_abnormal': 'Abnormal', 'functionally_normal': 'Normal', 'indeterminate': 'Indeterminate'},
             'PALB2_IGVF': {'functionally_abnormal': 'Abnormal', 'functionally_normal': 'Normal', 'indeterminate': 'Indeterminate'},
             'CTCF_IGVF': {'functionally_abnormal': 'Abnormal', 'functionally_normal': 'Normal', 'indeterminate': 'Indeterminate'},
             'RAD51D_IGVF': {'functionally_abnormal': 'Abnormal', 'functionally_normal': 'Normal', 'indeterminate': 'Indeterminate'},
             'SFPQ_IGVF': {'functionally_abnormal': 'Abnormal', 'functionally_normal': 'Normal', 'indeterminate': 'Indeterminate'},
             'XRCC2_IGVF': {'functionally_abnormal': 'Abnormal', 'functionally_normal': 'Normal', 'indeterminate': 'Indeterminate'},
             'PTEN_Matreyek_2018': {'functionally_abnormal': 'Abnormal', 'functionally_normal': 'Normal'},
             'F9_Popp_2025_model': {'WT-like': 'Normal','Loss of function': 'Abnormal'},
             'G6PD_IGVF': {'functionally_abnormal': 'Abnormal', 'functionally_normal': 'Normal'}, 
             'TSC2_IGVF': {'functionally_abnormal': 'Abnormal', 'functionally_normal': 'Normal'},
             'CARD11_Meitlis_2020_SGE_LoF': {'functional': 'Normal', 'not definitive': 'Indeterminate', 
                                                     'likely functional': 'Normal','likely nonfunctional': 'Abnormal', 
                                                     'nonfunctional': 'Abnormal'},
             'CARD11_Meitlis_2020_SGE_Ibrutinib_GoF': {'likely not gain of function': 'Normal', 'not definitive': 'Indeterminate',
                                                       'not gain of function': 'Normal', 'likely gain of function': 'Abnormal',
                                                      'gain of function': 'Abnormal'}}

In [30]:
def annotate_func_class(row):
    dataset = row['Dataset']

    # Original version: map authors' class names to categories.
    # label = row['auth_reported_func_class']
    # return func_class.get(dataset, {}).get(label)

    # New: use category from MaveDB, with simple mapping to this script's category names.
    category = row['auth_reported_func_class_category']
    if pd.isna(category):
        return None

    category_map = {
        'normal': 'Normal',
        'abnormal': 'Abnormal',
        'not_specified': 'Indeterminate'
    }
    return category_map.get(str(category).strip().lower(), None)

pp_2['StandardizedClass'] = pp_2.apply(annotate_func_class, axis=1)

In [31]:
#remove Flagged variants (incorrect or inconsistent mappings and splice variants in cDNA based assays (protein-level assays))

pd.set_option('display.max_columns', None)
pp_2 = pp_2[(pp_2['Flag'] != '*') & (pp_2['splice_var_amino'] != 'Yes')]

In [32]:
#load in Supplementary Data 3 (curation sheet)
mave = pd.read_excel(DATA_DIR / "Supplementary_Data/Supplementary_Data_3.xlsx", sheet_name="Curation", header=0)

In [33]:
#merge relevant columns with integrated variant effect dataset for downstream analysis

pp_mave = pd.merge(pp_2, mave[['Dataset Name','Score Intervals Reported?','Functional Classification Provided?']],
                   left_on = 'Dataset', right_on = 'Dataset Name', 
                   how = 'left')

In [34]:
#add in Scott et al. clinically relevant intervals reported in Scott et al. 2022 (PMID: 36550560)

Scott = (pp_mave['Dataset'] == 'MSH2_Scott_2022')

pp_mave.loc[Scott, [
    'Interval 1 Name', 'Interval 1 Range', 'Interval 1 Class'
]] = ['Normal', '(-inf, 0)', 'Normal']

pp_mave.loc[Scott, [
    'Interval 2 Name', 'Interval 2 Range', 'Interval 2 Class'
]] = ['Abnormal', '[0.4, inf)', 'Abnormal']


pp_mave.loc[Scott, 'Score Intervals Reported?'] = 'Reported'
pp_mave.loc[Scott, 'Functional Classification Provided?'] = 'No'

In [35]:
def parse_bound(bound_str):
    """
    Parse a string boundary from an interval definition into a numeric value.

    Handles special cases for infinity (e.g., '-inf', 'inf') and converts
    numeric strings to floats.
    """
    
    bound_str = bound_str.strip().lower()
    if bound_str in ['-inf', '-infinity']:
        return float('-inf')
    elif bound_str in ['inf', 'infinity', '+inf']:
        return float('inf')
    else:
        return float(bound_str)

def score_in_interval(score, interval):
    """
    Determine whether a numeric score falls within a specified interval.

    Intervals are defined as strings using set notation, e.g.:
        '(-inf, 0]', '(0, 0.4)', '[0.4, inf)'

    Inclusivity/exclusivity of bounds is inferred from brackets:
        '[' or ']'  -> inclusive
        '(' or ')'  -> exclusive
    """
    try:
        left = interval[0]
        right = interval[-1]
        lower_str, upper_str = interval[1:-1].split(',')

        lower = parse_bound(lower_str)
        upper = parse_bound(upper_str)

        lower_check = score >= lower if left == '[' else score > lower
        upper_check = score <= upper if right == ']' else score < upper

        return lower_check and upper_check
    except:
        return False

def find_matching_class(row, score):

    """
    Identify the functional class corresponding to a score for a given row.

    Iterates over interval definitions (Interval 1–9) and returns the first
    Class whose interval contains the score.
    """
    for i in range(1, 10): 
        interval_col = f'Interval {i} Range'
        class_col = f'Interval {i} Class'

        if interval_col in row and pd.notna(row[interval_col]):
            interval = row[interval_col]
            if score_in_interval(score, interval):
                return row[class_col]
    return 'Not specified'

In [36]:
#create a mask where datasets that don't have a functional classification reported and have score intervals reported are used 

mask = (
    (pp_mave['Functional Classification Provided?'].isin(['No'])) &
    (pp_mave['Score Intervals Reported?'] == 'Reported')
)

pp_mave.loc[mask, 'StandardizedClass'] = pp_mave.loc[mask].apply(
    lambda row: find_matching_class(row, row['auth_reported_score']),
    axis=1
)

In [37]:
import numpy as np
priority_genes = ['BRCA1', 'PTEN', 'MSH2', 'TP53']

#create new column with clinvar significance for genes where 2018 calibrations are needed, and if not then 2025 
pp_mave['clinvar_18_25'] = np.where(
    pp_mave['Gene'].isin(priority_genes),  
    pp_mave['clinvar_sig_2018'], 
    pp_mave['clinvar_sig_2025']
)


In [38]:
# Define your groups
pathogenic_group = {
    "Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic"
}
benign_group = {
    "Benign", "Likely benign", "Benign/Likely benign"
}
conflicts_group = {"Conflicting classifications of pathogenicity"}

def summarize_clnsig(series):
    sigs = set(series.dropna())  # unique values, ignore NaN/None

    # If everything is NA → return blank
    if len(sigs) == 0:
        return np.nan

    # Explicit flags
    has_path = any(val in pathogenic_group for val in sigs)
    has_benign = any(val in benign_group for val in sigs)
    has_conflict = any(val in conflicts_group for val in sigs)
    has_uncertain = "Uncertain significance" in sigs

    # Conflict rules
    if has_conflict or (has_path and has_benign):
        return "has clinvar conflict"

    # Pathogenic-only group
    if has_path:
        if "Likely pathogenic" in sigs or "Pathogenic/Likely pathogenic" in sigs:
            return "Likely pathogenic"
        return "Pathogenic"

    # Benign-only group
    if has_benign:
        if "Likely benign" in sigs or "Benign/Likely benign" in sigs:
            return "Likely benign"
        return "Benign"

    # Uncertain-only group
    if has_uncertain and len(sigs) == 1:
        return "Uncertain significance"

    # If it's a mix of uncertain with something else → conflict
    if has_uncertain:
        return "VUS/conflict"

    if len(sigs) == 1:
        return next(iter(sigs))   # grab the single element from the set
    else:
        return "multiple other classifications"

pp_mave["clnsig_group_18_25"] = (
    # JS 20260713
    # pp_mave.groupby('ID')["clinvar_18_25"]
    pp_mave.groupby('mavedb_variant_urn')["clinvar_18_25"]
    .transform(summarize_clnsig)
)

In [39]:
#standardize the case for StandardizedClass
pp_mave['StandardizedClass'] = pp_mave['StandardizedClass'].str.upper().fillna(pp_mave['StandardizedClass'])

In [40]:
#subset to datasets wheere score intervals are reported or functional classifications for variants is reported 

pp_mave = pp_mave[
    (pp_mave['Score Intervals Reported?'] == 'Reported')|
    (pp_mave['Functional Classification Provided?'] == 'Yes')
]

In [41]:
import pandas as pd

col_header = "StandardizedClass"
functional = "NORMAL"
abnormal_func = "ABNORMAL"

pathogenic_group = {
    "Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic"
}
benign_group = {
    "Benign", "Likely benign", "Benign/Likely benign"
}


def Odds_LR(df, clnsig_group):
    try:
        pathogenic = df[df[clnsig_group].isin(pathogenic_group)]
        benign = df[df[clnsig_group].isin(benign_group)]

        # Count controls
        pathogenic_controls = len(pathogenic)
        benign_controls = len(benign)
        total_controls = pathogenic_controls + benign_controls

        prior_prob = pathogenic_controls/total_controls

        if total_controls == 0:
            return {
                "Dataset": df["Dataset"].iloc[0],
                "Total Controls": 0,
                "OddsNormal": "No controls",
                "OddsAbnormal": "No controls",
                "Pathogenic Controls": 0,
                "Benign Controls": 0,
                "Prior Probability Pathogenic": 0,  
                "Total Assay Abnormal": 0,
                "True Path in Abnormal":0,
                "Total Assay Normal": 0,
                "True Path in Normal":0,
                "Pseudocount Details": "No controls"
            }

        pseudo_reasons = []

        # Pathogenic variants by functional class
        path_abnormal = len(pathogenic[pathogenic[col_header] == abnormal_func])
        path_normal = len(pathogenic[pathogenic[col_header] == functional])

        # Benign variants by functional class
        benign_abnormal = len(benign[benign[col_header] == abnormal_func])
        benign_normal = len(benign[benign[col_header] == functional])

        # Total in each functional class
        assay_abnormal_num = len(df[df[col_header] == abnormal_func])
        assay_normal_num = len(df[df[col_header] == functional])

        # ---- PSEUDOCOUNT ADJUSTMENTS ----

        # For LR+: need benign_abnormal > 0
        if benign_abnormal == 0:
            benign_abnormal += 1
            pseudo_reasons.append("Added 1 misclassified benign variant to abnormal function class")

        # For LR-: need path_normal > 0
        if path_normal == 0:
            path_normal += 1
            pseudo_reasons.append("Added 1 misclassified pathogenic variant to normal function class")

        # ---- CALCULATE LRs ----

        # Check for sufficient data
        if benign_controls == 0:
            return {
                "Dataset": df["Dataset"].iloc[0],
                "Total Controls": total_controls,
                "OddsNormal": "No benign controls",
                "OddsAbnormal": "No benign controls",
                "Pathogenic Controls": pathogenic_controls,
                "Benign Controls": benign_controls,
                "Prior Probability Pathogenic": prior_prob,  
                "Total Assay Abnormal": assay_abnormal_num,
                "True Path in Abnormal":path_abnormal,
                "Total Assay Normal": assay_normal_num,
                "True Path in Normal": path_normal,
                "Pseudocount Details": "No benign controls"
            }
  

        if pathogenic_controls == 0:
            return {
                "Dataset": df["Dataset"].iloc[0],
                "Total Controls": total_controls,
                "OddsNormal": "No pathogenic controls",
                "OddsAbnormal": "No pathogenic controls",
                "Pathogenic Controls": pathogenic_controls,
                "Benign Controls": benign_controls,
                "Prior Probability Pathogenic": prior_prob,  
                "Total Assay Abnormal": assay_abnormal_num,
                "True Path in Abnormal":path_abnormal,
                "Total Assay Normal": assay_normal_num,
                "True Path in Normal": path_normal,
                "Pseudocount Details": "No pathogenic controls"
            }

        # LR+ (for abnormal result)
        if assay_abnormal_num == 0:
            OddsAbnormal = "No functionally abnormal controls"
        else:
            OddsAbnormal = (path_abnormal / pathogenic_controls) / (benign_abnormal / benign_controls)

        # LR- (for normal result)
        if assay_normal_num == 0:
            OddsNormal = "No functionally normal controls"
        else:
            OddsNormal = (path_normal / pathogenic_controls) / (benign_normal / benign_controls)

        return {
            "Dataset": df["Dataset"].iloc[0],
            "Total Controls": total_controls,
            "OddsNormal": round(OddsNormal, 4) if isinstance(OddsNormal, float) else OddsNormal,
            "OddsAbnormal": round(OddsAbnormal, 4) if isinstance(OddsAbnormal, float) else OddsAbnormal,
            "Pathogenic Controls": pathogenic_controls,
            "Benign Controls": benign_controls,
            "Prior Probability Pathogenic": prior_prob,  
            "Total Assay Abnormal": assay_abnormal_num,
            "True Path in Abnormal":path_abnormal,
            "Total Assay Normal": assay_normal_num,
            "True Path in Normal": path_normal,
            "Pseudocount Details": "; ".join(pseudo_reasons) if pseudo_reasons else "None"
        }

        
    except Exception as e:
        return {
            "Dataset": df["Dataset"].iloc[0] if len(df) > 0 else "Unknown",
            "Error": str(e)
        }

In [42]:
# JS 20260713
# pp_drop = pp_mave.drop_duplicates('ID')
pp_drop = pp_mave.drop_duplicates('mavedb_variant_urn')

In [43]:
all_vars = pd.DataFrame([
    Odds_LR(group, 'clnsig_group_18_25')
    for _, group in pp_drop.groupby("Dataset")
])

In [44]:
for col in ["OddsAbnormal", "OddsNormal","Prior Probability Pathogenic"]:
    all_vars[col] = (
        pd.to_numeric(all_vars[col], errors="coerce")
        .round(4)
        .combine_first(all_vars[col])
    )

In [45]:
import pandas as pd
import numpy as np

def odds_to_evidence(x):
    """
    Map OddsPath value to ACMG/ClinGen evidence strength.

    Non-numeric or non-calculable values return 'Not available'.
    """
    # Handle missing / non-numeric
    if pd.isna(x):
        return "Not available"
    if isinstance(x, str):
        return "Not available"

    try:
        x = float(x)
    except:
        return "Not available"

    # Benign evidence
    if x < 0.053:
        return "BS3_strong"
    elif x < 0.23:
        return "BS3_moderate"
    elif x < 0.48:
        return "BS3_supporting"

    # Indeterminate
    elif 0.48 <= x <= 2.1:
        return "Indeterminate"

    # Pathogenic evidence
    elif x <= 4.3:
        return "PS3_supporting"
    elif x <= 18.7:
        return "PS3_moderate"
    elif x <= 350:
        return "PS3_strong"
    else:
        return "PS3_very_strong"


In [46]:
all_vars["Evidence Code Normal"]   = all_vars["OddsNormal"].apply(odds_to_evidence)
all_vars["Evidence Code Abnormal"] = all_vars["OddsAbnormal"].apply(odds_to_evidence)

In [47]:
all_vars = all_vars[all_vars['Dataset'] != 'SFPQ_IGVF']

In [48]:
all_vars = all_vars[~all_vars['Dataset'].isin(['F9_Popp_2025_carboxy_F9_specific',
       'F9_Popp_2025_heavy_chain', 'F9_Popp_2025_light_chain', 'F9_Popp_2025_strep_2'])]

In [49]:
all_vars.to_csv(OUT_DIR/"OddsPath_calibrations.csv.gz", compression = 'gzip', index = None)